In [1]:
!test -f temp.bufr \
    || wget https://sites.ecmwf.int/repository/pdbufr/test-data/temp.bufr \
             --output-document=temp.bufr

# Generic vs flat reader

In [2]:
import pdbufr

## Generic reader: one row per pressure level

The generic reader traverses the hierarchical BUFR structure.  For each pressure
level where all requested keys are available it emits one row.  This gives a
"long" (tidy) format that is convenient for filtering and plotting single
variables across levels.

In [3]:
df_generic = pdbufr.read_bufr(
    "temp.bufr",
    columns=(
        "WMO_station_id",
        "latitude",
        "longitude",
        "data_datetime",
        "pressure",
        "airTemperature",
        "windDirection",
        "windSpeed",
    ),
)
print(f"{len(df_generic)} rows – one per pressure level across all profiles")
df_generic.head()

26005 rows – one per pressure level across all profiles


,latitude,longitude,pressure,airTemperature,windDirection,windSpeed,data_datetime,WMO_station_id
0,58.47,-78.08,100300.0,258.3,NaN,NaN,2008-12-08 12:00:00,71907
1,58.47,-78.08,100000.0,259.7,0.0,0.0,2008-12-08 12:00:00,71907
2,58.47,-78.08,99800.0,261.1,NaN,NaN,2008-12-08 12:00:00,71907
3,58.47,-78.08,99100.0,261.7,NaN,NaN,2008-12-08 12:00:00,71907
4,58.47,-78.08,92500.0,258.1,275.0,5.0,2008-12-08 12:00:00,71907


## Generic reader: ranked keys are not supported

The generic reader documentation explicitly states that ranks cannot be used in
key names.  Passing a ranked key (e.g. ``"#1#pressure"``) returns an empty
DataFrame because the reader's hierarchical collector cannot resolve the rank
prefix.

In [4]:
df_ranked = pdbufr.read_bufr(
    "temp.bufr",
    columns=("#1#pressure", "airTemperature"),  # ranked key – not supported
)
print(f"{len(df_ranked)} rows, columns: {list(df_ranked.columns)}")

0 rows, columns: []


## Flat reader (block extraction): one row per profile

With ``columns="all"`` the flat reader extracts every key in a message as a
single wide row.  Repeated variables become ranked columns:
``#1#pressure``, ``#2#pressure``, …, ``#N#pressure``.

This gives one row per radiosonde launch – a "wide" format where each column
corresponds to one measurement at one level.

In [5]:
import warnings

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", module="pdbufr")
    df_flat_block = pdbufr.read_bufr(
        "temp.bufr",
        columns="all",
        reader="flat",
    )

pressure_cols = [c for c in df_flat_block.columns if "pressure" in c]
print(f"{len(df_flat_block)} rows – one per profile")
print(f"{len(df_flat_block.columns)} total columns, {len(pressure_cols)} pressure levels")
print("Pressure columns:", pressure_cols[:5], "...")

420 rows – one per profile
1161 total columns, 157 pressure levels
Pressure columns: ['#1#pressure', '#2#pressure', '#3#pressure', '#4#pressure', '#5#pressure'] ...


In [6]:
df_flat_ind = pdbufr.read_bufr(
    "temp.bufr",
    columns=[
        "WMO_station_id",
        "latitude",
        "longitude",
        "data_datetime",
        "#1#pressure",
        "#1#airTemperature",
        "#1#windSpeed",
        "#2#pressure",
        "#2#airTemperature",
        "#2#windSpeed",
        "#3#pressure",
        "#3#airTemperature",
        "#3#windSpeed",
    ],
    reader="flat",
)
print(f"{len(df_flat_ind)} rows – one per profile, {len(df_flat_ind.columns)} columns")
df_flat_ind.head()

417 rows – one per profile, 13 columns


,WMO_station_id,latitude,longitude,data_datetime,#1#pressure,#1#airTemperature,#1#windSpeed,#2#pressure,#2#airTemperature,#2#windSpeed,#3#pressure,#3#airTemperature,#3#windSpeed
0,71907,58.47,-78.08,2008-12-08 12:00:00,100300.0,258.3,NaN,100000.0,259.7,0.0,99800.0,261.1,NaN
1,71823,53.75,-73.67,2008-12-08 12:00:00,100000.0,NaN,NaN,97400.0,256.7,3.0,93700.0,255.1,NaN
2,89009,-90.00,0.00,2008-12-08 12:00:00,100000.0,NaN,NaN,92500.0,NaN,NaN,85000.0,NaN,NaN
3,78486,18.43,-69.88,2008-12-08 12:00:00,101600.0,294.6,3.0,100000.0,295.0,5.0,99600.0,295.0,NaN
4,91165,21.98,-159.33,2008-12-08 12:00:00,101500.0,293.8,3.0,100900.0,296.4,NaN,100000.0,296.2,NaN


## Summary

| | Generic reader | Flat reader |
|---|---|---|
| **Result shape** | One row per pressure level (long format) | One row per message / profile (wide format) |
| **Ranked keys** | Not supported – returns empty DataFrame | First-class: ``#1#pressure``, ``#2#pressure``, … |
| **Typical use** | Filtering/plotting single variables across levels | Keeping full profiles as single records |
| **Column count** | Equal to number of requested keys | Up to one column per rank per key |